In [ ]:
import numpy as np
import time
import multiprocessing
import matplotlib.pyplot as plt
import seaborn as sns

# 1. DEFINISI FUNGSI BEBAN KERJA (WORKLOAD)
# Fungsi ini disimulasikan sebagai filter deteksi gambar infrastruktur yang memakan banyak tenaga CPU
def process_infrastructure_image(image_id):
    """
    Mensimulasikan pemrosesan 1 citra resolusi tinggi (2000x2000 piksel).
    Memanfaatkan operasi trigonometri untuk memaksa CPU bekerja ekstra (CPU-bound task).
    """
    np.random.seed(image_id)
    # Membuat matriks 2000 x 2000
    matrix = np.random.rand(2000, 2000)
    
    # Simulasi perhitungan piksel berat (misal: filtering, edge detection)
    for _ in range(3):
        matrix = np.sin(matrix) ** 2 + np.cos(matrix) ** 2 + np.tan(matrix / 2)
        
    return np.mean(matrix)

if __name__ == '__main__':
    print("=== BENCHMARK HIGH PERFORMANCE COMPUTING (HPC) ===")
    
    # 2. SETUP PENGUJIAN
    jumlah_citra = 100 # Mensimulasikan antrean 100 foto drone
    daftar_citra = list(range(jumlah_citra))
    jumlah_core = multiprocessing.cpu_count()
    
    print(f"Total Citra yang diproses : {jumlah_citra} file")
    print(f"Total CPU Core terdeteksi : {jumlah_core} Cores (Pekerja)\n")
    
    # 3. PENGUJIAN 1: EKSEKUSI SEKUENSIAL (SERIAL)
    print("[1/2] Memulai komputasi Serial (Konvensional - 1 Core)...")
    start_serial = time.time()
    
    # Dieksekusi satu per satu dalam loop biasa
    hasil_serial = [process_infrastructure_image(img) for img in daftar_citra]
    
    end_serial = time.time()
    waktu_serial = end_serial - start_serial
    print(f"✅ Selesai! Waktu Serial: {waktu_serial:.2f} detik\n")
    
    # 4. PENGUJIAN 2: EKSEKUSI PARALEL (HPC)
    print(f"[2/2] Memulai komputasi Paralel (HPC - Membagi tugas ke {jumlah_core} Core)...")
    start_parallel = time.time()
    
    # Membuka pool pekerja sebanyak jumlah core, lalu membagi tugas secara merata
    with multiprocessing.Pool(processes=jumlah_core) as pool:
        hasil_parallel = pool.map(process_infrastructure_image, daftar_citra)
        
    end_parallel = time.time()
    waktu_parallel = end_parallel - start_parallel
    print(f"✅ Selesai! Waktu Paralel: {waktu_parallel:.2f} detik\n")
    
    # 5. KALKULASI METRIK PERFORMA
    speedup = waktu_serial / waktu_parallel
    efisiensi = (speedup / jumlah_core) * 100
    
    print("=== KESIMPULAN PERFORMA EKSEKUSI ===")
    print(f"⏱️ Penghematan Waktu : {(waktu_serial - waktu_parallel):.2f} detik")
    print(f"🚀 Speedup Factor    : {speedup:.2f}x Lebih Cepat")
    print(f"⚙️ Efisiensi CPU     : {efisiensi:.1f}%")
    
    # 6. VISUALISASI HASIL BENCHMARK
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(8, 5))
    
    metrik_label = ['Serial (1 Core)', f'Paralel ({jumlah_core} Cores)']
    waktu_eksekusi = [waktu_serial, waktu_parallel]
    
    ax = sns.barplot(x=metrik_label, y=waktu_eksekusi, palette=['#e74c3c', '#2ecc71'])
    plt.title('Perbandingan Waktu Eksekusi Pemrosesan Citra\n(Lebih Pendek = Lebih Baik)', fontweight='bold', pad=15)
    plt.ylabel('Waktu Tempuh (Detik)')
    plt.xlabel('Metode Eksekusi')
    
    # Menambahkan anotasi angka di atas bar
    for i, v in enumerate(waktu_eksekusi):
        ax.text(i, v + 0.5, f"{v:.2f} dtk", ha='center', va='bottom', fontweight='bold')
        
    # Menambahkan teks Speedup
    plt.annotate(f"Speedup: {speedup:.2f}x", 
                 xy=(1, waktu_parallel), 
                 xytext=(0.5, waktu_serial * 0.8),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
                 fontsize=12, fontweight='bold', color='#27ae60', ha='center')

    plt.tight_layout()
    plt.show()